# Notebook 04: Deep Error Analysis (FP vs. FN Investigation)
## Fraud Detection & Anomaly Analytics System

This notebook breaks down model errors into False Positives (customer friction) and False Negatives (financial loss), profiles behavioral characteristics across prediction cohorts, and investigates vulnerability by fraud scenario.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.error_analysis import perform_error_analysis
%matplotlib inline


### 1. Run Complete Error Analysis on Validation Cohorts

In [ ]:
results = perform_error_analysis(
    data_path='../data/processed/val.csv',
    probas_path='../outputs/predictions/val_probas_E3.npy',
    optimal_threshold=0.20,
    output_dir='../outputs/plots',
    metrics_dir='../outputs/metrics'
)


### 2. Key Empirical Insights from Error Analysis

#### A. False Positives (Innocent Flagged - 35 transactions):
- **Mean Amount**: **.25** (vs. legitimate baseline of .75).
- **Customer Amount Ratio**: **2.52x**.
- **Root Cause**: The model falsely alarms on legitimate cardholders making rare, unusually large purchases (e.g. appliances or travel).

#### B. False Negatives (Fraud Missed - 582 transactions):
- **Mean Amount**: **.19** (virtually identical to normal spend of .75!).
- **Customer Amount Ratio**: **1.12x**.
- **Root Cause (Scenario 2 Vulnerability)**: 540 of the 582 missed frauds (92.8%) belong to **Scenario 2 (Compromised Terminals)**. Because amounts and cardholder behavior look completely ordinary, a supervised model without terminal risk features cannot detect them.

#### C. Architectural Solution:
This explains why **Phase 9 (Isolation Forest)** and our **Phase 12 (Fraud Risk Engine)** are crucial: combining supervised probability with terminal anomaly scoring captures attacks that look superficially normal.